# FounderEndurance-v1 — Custom Gymnasium RL Environment

**Meta PyTorch OpenEnv Hackathon Submission**

A startup-founder survival simulator where an RL agent must balance:
- **Health** (sleep debt, cortisol, caffeine toxicity)
- **Revenue** (cash runway, fundraising)
- **Morale** (team morale, death-march decay)
- **Velocity** (product velocity, market conditions)

Over a 90-day episode leading up to product launch.

---

### Notebook Sections
1. Install Dependencies
2. Scaffold Project Files
3. Install Package
4. Run Tests (Gymnasium Compliance)
5. Train PPO Agent
6. Evaluate & Visualise Results
7. Launch TensorBoard

## 1. Install Dependencies

In [ ]:
!pip install gymnasium>=0.29.0 numpy>=1.24.0 torch>=2.1.0 stable-baselines3>=2.2.0 tensorboard>=2.14.0 pytest matplotlib

## 2. Scaffold Project Files

Create the entire project structure directly in Colab.

In [ ]:
import os

# Create directory structure
for d in [
    "founder_endurance/envs",
    "founder_endurance/utils",
    "train",
    "tests",
    "models/best",
    "logs",
    "tb_logs",
]:
    os.makedirs(d, exist_ok=True)

print("Directory structure created.")

In [ ]:
%%writefile founder_endurance/utils/__init__.py
# founder_endurance/utils/__init__.py

In [ ]:
%%writefile founder_endurance/envs/founder_env.py
# founder_endurance/envs/founder_env.py
import gymnasium as gym
from gymnasium import spaces
import numpy as np


class FounderEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 1}

    # Reward weights (tunable hyperparameters)
    W1 = 1.0   # cash_runway delta weight
    W2 = 0.5   # product_velocity delta weight
    W3 = 2.0   # health penalty weight

    WORK_HOURS = [4, 8, 12, 16]  # indexed by work_hours_idx

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode

        # Observation: 10 normalised floats in [0, 1]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(10,), dtype=np.float32
        )

        # Action: [work_hours(4), focus(4), health(3)]
        self.action_space = spaces.MultiDiscrete([4, 4, 3])

        # Internal counters (not part of obs)
        self._obs_labels = [
            "sleep_debt", "cortisol_level", "caffeine_toxicity",
            "product_velocity", "team_morale", "cash_runway",
            "market_condition", "active_crisis", "day_of_week", "days_to_launch",
        ]
        self._consecutive_overwork = 0
        self._caffeine_clearance_days = 0
        self._day = 0
        self._prev_obs = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self._day = 0
        self._consecutive_overwork = 0
        self._caffeine_clearance_days = 0

        # Initial state -- founder is reasonably healthy at episode start
        obs = np.array([
            0.10,  # sleep_debt
            0.15,  # cortisol_level
            0.00,  # caffeine_toxicity
            0.50,  # product_velocity
            0.80,  # team_morale
            0.60,  # cash_runway
            0.50,  # market_condition (mid-cycle)
            0.00,  # active_crisis
            0.00,  # day_of_week (Monday)
            1.00,  # days_to_launch (full 90 days)
        ], dtype=np.float32)

        self._prev_obs = obs.copy()
        info = {}
        return obs, info

    def step(self, action):
        assert self.action_space.contains(action), f"Invalid action: {action}"

        work_hours_idx, focus_idx, health_idx = action
        obs = self._prev_obs.copy()

        # 1. Apply work-hours effect on sleep_debt
        obs = self._apply_work_and_sleep(obs, work_hours_idx, health_idx)

        # 2. Apply focus-area effect on company variables
        obs = self._apply_focus(obs, focus_idx, work_hours_idx)

        # 3. Apply death-march morale decay (non-linear)
        obs = self._apply_morale_decay(obs, work_hours_idx)

        # 4. Stochastic crisis generation / resolution
        obs = self._apply_crisis(obs, focus_idx)

        # 5. Advance time variables
        self._day += 1
        obs[8] = (self._day % 7) / 7.0
        obs[9] = max(0.0, (90 - self._day) / 90.0)

        # 6. Clip all values to [0, 1]
        obs = np.clip(obs, 0.0, 1.0)

        # 7. Compute reward
        reward, terminated = self._compute_reward(obs)

        truncated = self._day >= 90

        if truncated and not terminated:
            # Check for successful launch
            if obs[3] > 0.8 and obs[5] > 0.0:
                reward += 500.0

        self._prev_obs = obs.copy()

        info = {"day": self._day, "action": action.tolist()}

        return obs, float(reward), terminated, truncated, info

    def render(self):
        if self.render_mode is None:
            return None
        if self.render_mode == "human":
            if self._prev_obs is not None:
                print(f"Day {self._day}")
                for i, label in enumerate(self._obs_labels):
                    print(f"  {label}: {self._prev_obs[i]:.3f}")
            return None
        if self.render_mode == "rgb_array":
            # Return a minimal placeholder image
            img = np.zeros((64, 64, 3), dtype=np.uint8)
            return img

    # ------------------------------------------------------------------
    # Dynamics Subsystem 1: Caffeine & Sleep Debt
    # ------------------------------------------------------------------
    def _apply_work_and_sleep(self, obs, work_hours_idx, health_idx):
        hours = self.WORK_HOURS[work_hours_idx]
        using_caffeine = (health_idx == 1)
        using_therapy = (health_idx == 2)

        # Therapy hard-caps work hours at 8
        if using_therapy:
            hours = min(hours, 8)
            obs[1] -= 0.20  # cortisol_level drops sharply

        # Caffeine toxicity
        if using_caffeine:
            obs[2] += 0.20  # caffeine_toxicity
            self._caffeine_clearance_days = 0
        else:
            self._caffeine_clearance_days += 1
            if self._caffeine_clearance_days >= 3:
                obs[2] -= 0.10  # caffeine clears after 3 days

        # Sleep debt accumulation / recovery
        sleep_hours = 24 - hours
        if sleep_hours < 7:
            debt_increase = (7 - sleep_hours) / 7.0 * 0.15
            obs[0] += debt_increase
        else:
            # Can only clear sleep debt if caffeine_toxicity is low
            if obs[2] <= 0.6:
                obs[0] -= 0.05  # gradual recovery

        # Cortisol increases with long hours
        if hours >= 12:
            obs[1] += 0.05 * (hours / 8.0)

        return obs

    # ------------------------------------------------------------------
    # Dynamics Subsystem 2: Focus Area Effects
    # ------------------------------------------------------------------
    def _apply_focus(self, obs, focus_idx, work_hours_idx):
        # Effectiveness multiplier: halved if caffeine_toxicity > 0.6
        effectiveness = 0.5 if obs[2] > 0.6 else 1.0
        hours = self.WORK_HOURS[work_hours_idx]
        intensity = (hours / 16.0) * effectiveness

        if focus_idx == 0:    # Product / Engineering
            obs[3] += 0.08 * intensity    # product_velocity up
            # Crisis resolved faster when team focuses on product
            if obs[7] > 0.0:
                obs[7] -= 0.3 * intensity

        elif focus_idx == 1:  # Fundraising / Sales
            # Market condition scales the cash gain
            cash_gain = 0.10 * intensity * obs[6]
            obs[5] += cash_gain

        elif focus_idx == 2:  # Team Building
            obs[4] += 0.10 * intensity    # team_morale up
            obs[3] -= 0.03               # velocity dips temporarily

        elif focus_idx == 3:  # Firefighting
            obs[1] -= 0.08 * intensity    # cortisol_level down

        # Cash runway always decays by a daily burn rate
        obs[5] -= 0.01

        # Market condition: stochastic sine wave
        # Advance phase by a random small step each day
        phase_delta = self.np_random.uniform(0.02, 0.06)
        obs[6] = (np.sin(self._day * phase_delta) + 1.0) / 2.0

        return obs

    # ------------------------------------------------------------------
    # Dynamics Subsystem 3: Death March (Non-Linear Morale Decay)
    # ------------------------------------------------------------------
    def _apply_morale_decay(self, obs, work_hours_idx):
        hours = self.WORK_HOURS[work_hours_idx]

        if hours > 12:
            self._consecutive_overwork += 1
        else:
            self._consecutive_overwork = max(0, self._consecutive_overwork - 1)

        if self._consecutive_overwork > 3:
            decay = 0.05 * (self._consecutive_overwork ** 1.5)
            obs[4] -= decay  # exponential morale loss

        # Hard cap: quiet-quitting kicks in at low morale
        if obs[4] < 0.2:
            obs[3] = min(obs[3], 0.10)  # product_velocity capped at 10%

        return obs

    # ------------------------------------------------------------------
    # Dynamics Subsystem 4: Stochastic Crisis Engine
    # ------------------------------------------------------------------
    def _apply_crisis(self, obs, focus_idx):
        # Crisis resolution first
        if obs[7] > 0.0:
            obs[1] += 0.05  # active crisis raises cortisol each day

        # Stochastic crisis generation
        p_crisis = 0.05 + (1.0 - obs[3]) * 0.15 + obs[2] * 0.10
        if obs[7] == 0.0 and self.np_random.random() < p_crisis:
            obs[7] = 1.0  # new crisis activated

        return obs

    # ------------------------------------------------------------------
    # Reward Function
    # ------------------------------------------------------------------
    def _compute_reward(self, obs):
        prev = self._prev_obs
        reward = 0.0
        terminated = False

        # Delta rewards
        delta_cash = obs[5] - prev[5]
        delta_velocity = obs[3] - prev[3]
        reward += self.W1 * delta_cash
        reward += self.W2 * delta_velocity

        # Health penalties
        if obs[1] > 0.8:    # cortisol_level
            reward -= 1.0
        if obs[0] > 0.8:    # sleep_debt
            reward -= 1.5

        # Terminal: Hospitalisation
        if obs[1] >= 1.0 or obs[0] >= 1.0:
            reward -= 250.0
            terminated = True

        # Terminal: Bankruptcy
        if obs[5] <= 0.0:
            reward -= 250.0
            terminated = True

        # Terminal: Mutiny
        if obs[4] <= 0.0:
            reward -= 200.0
            terminated = True

        return reward, terminated

In [ ]:
%%writefile founder_endurance/envs/__init__.py
from founder_endurance.envs.founder_env import FounderEnv

__all__ = ["FounderEnv"]

In [ ]:
%%writefile founder_endurance/__init__.py
from gymnasium.envs.registration import register

register(
    id="FounderEndurance-v1",
    entry_point="founder_endurance.envs.founder_env:FounderEnv",
    max_episode_steps=90,
    reward_threshold=400.0,
)

In [ ]:
%%writefile setup.py
from setuptools import setup, find_packages

setup(
    name="founder_endurance",
    version="0.1.0",
    packages=find_packages(),
    install_requires=[
        "gymnasium>=0.29.0",
        "numpy>=1.24.0",
    ],
    entry_points={
        "gymnasium.envs": [
            "founder_endurance/FounderEndurance-v1=founder_endurance.envs:FounderEnv",
        ]
    },
)

In [ ]:
%%writefile tests/test_env.py
# tests/test_env.py
import pytest
import numpy as np
import gymnasium as gym
from gymnasium.utils.env_checker import check_env
import founder_endurance


@pytest.fixture
def env():
    e = gym.make("FounderEndurance-v1")
    yield e
    e.close()


def test_gym_compliance(env):
    """check_env validates spaces, step(), reset() contracts."""
    check_env(env.unwrapped)


def test_reset_returns_valid_obs(env):
    obs, info = env.reset()
    assert obs.shape == (10,)
    assert np.all(obs >= 0.0) and np.all(obs <= 1.0)


def test_step_shapes(env):
    env.reset()
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    assert obs.shape == (10,)
    assert isinstance(reward, float)
    assert isinstance(terminated, bool)


def test_bankruptcy_terminates(env):
    """Force cash_runway to 0 and verify terminal state."""
    obs, _ = env.reset()
    env.unwrapped._prev_obs[5] = 0.001  # near-zero cash
    # All-nighter + Fundraising in bear market
    for _ in range(5):
        _, _, terminated, _, _ = env.step(np.array([3, 1, 0]))
        if terminated:
            break
    assert terminated


def test_deterministic_with_seed(env):
    obs1, _ = env.reset(seed=42)
    obs2, _ = env.reset(seed=42)
    np.testing.assert_array_equal(obs1, obs2)

In [ ]:
%%writefile train/train_ppo.py
# train/train_ppo.py
import gymnasium as gym
import founder_endurance  # triggers registration
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback


def main():
    env_id = "FounderEndurance-v1"

    # Vectorised training: 8 parallel environments
    train_env = make_vec_env(env_id, n_envs=8)
    eval_env = make_vec_env(env_id, n_envs=2)

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path="./models/best/",
        log_path="./logs/",
        eval_freq=5000,
        deterministic=True,
        render=False,
    )

    model = PPO(
        "MlpPolicy",
        train_env,
        verbose=1,
        tensorboard_log="./tb_logs/",
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
    )

    model.learn(
        total_timesteps=1_000_000,
        callback=eval_callback,
    )

    model.save("./models/founder_ppo_final")


if __name__ == "__main__":
    main()

## 3. Install Package (editable mode)

In [ ]:
!pip install -e . -q

## 4. Run Tests (Gymnasium Compliance)

In [ ]:
!python -m pytest tests/test_env.py -v

## 5. Quick Smoke Test (manual episode rollout)

In [ ]:
import gymnasium as gym
import numpy as np
import founder_endurance

env = gym.make("FounderEndurance-v1")
obs, info = env.reset(seed=42)

total_reward = 0.0
steps = 0

while True:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    if terminated or truncated:
        break

print(f"Episode finished in {steps} steps")
print(f"Total reward: {total_reward:.2f}")
print(f"Terminated: {terminated} | Truncated: {truncated}")
print(f"Final obs: {np.round(obs, 3)}")
env.close()

## 6. Train PPO Agent

Training with 500K timesteps (reduced for Colab). Increase `total_timesteps` for better results.

In [ ]:
import gymnasium as gym
import founder_endurance
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

env_id = "FounderEndurance-v1"

# Vectorised training: 8 parallel environments
train_env = make_vec_env(env_id, n_envs=8)
eval_env = make_vec_env(env_id, n_envs=2)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./models/best/",
    log_path="./logs/",
    eval_freq=5000,
    deterministic=True,
    render=False,
)

model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    tensorboard_log="./tb_logs/",
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
)

print("Starting PPO training...")
model.learn(
    total_timesteps=500_000,
    callback=eval_callback,
)

model.save("./models/founder_ppo_final")
print("Training complete! Model saved to ./models/founder_ppo_final")

## 7. Evaluate Trained Agent

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load the best model
best_model = PPO.load("./models/best/best_model")

env = gym.make("FounderEndurance-v1")

# Run 20 evaluation episodes
n_eval = 20
episode_rewards = []
episode_lengths = []
termination_reasons = {"launch": 0, "hospitalized": 0, "bankrupt": 0, "mutiny": 0, "survived": 0}

for ep in range(n_eval):
    obs, _ = env.reset(seed=ep)
    total_reward = 0.0
    steps = 0
    done = False

    while not done:
        action, _ = best_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

    # Classify termination
    if truncated and not terminated:
        if obs[3] > 0.8 and obs[5] > 0.0:
            termination_reasons["launch"] += 1
        else:
            termination_reasons["survived"] += 1
    elif obs[1] >= 1.0 or obs[0] >= 1.0:
        termination_reasons["hospitalized"] += 1
    elif obs[5] <= 0.0:
        termination_reasons["bankrupt"] += 1
    elif obs[4] <= 0.0:
        termination_reasons["mutiny"] += 1

env.close()

print(f"=== Evaluation over {n_eval} episodes ===")
print(f"Mean reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
print(f"Mean length: {np.mean(episode_lengths):.1f} / 90 days")
print(f"\nTermination breakdown:")
for reason, count in termination_reasons.items():
    print(f"  {reason}: {count}/{n_eval}")

In [ ]:
# Plot episode rewards
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(n_eval), episode_rewards, color="steelblue", alpha=0.8)
axes[0].axhline(y=np.mean(episode_rewards), color="red", linestyle="--", label=f"Mean: {np.mean(episode_rewards):.1f}")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].set_title("Episode Rewards (Trained PPO Agent)")
axes[0].legend()

# Termination pie chart
labels = [k for k, v in termination_reasons.items() if v > 0]
sizes = [v for v in termination_reasons.values() if v > 0]
colors = ["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6", "#3498db"]
axes[1].pie(sizes, labels=labels, autopct="%1.0f%%", colors=colors[:len(labels)])
axes[1].set_title("Termination Reasons")

plt.tight_layout()
plt.show()

## 8. Single Episode Trajectory Visualisation

In [ ]:
# Run one detailed episode and plot all observation dimensions over time
env = gym.make("FounderEndurance-v1")
obs, _ = env.reset(seed=7)

obs_labels = [
    "sleep_debt", "cortisol", "caffeine_tox",
    "velocity", "morale", "cash",
    "market", "crisis", "day_of_week", "days_left",
]

trajectory = [obs.copy()]
rewards = []
done = False

while not done:
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    trajectory.append(obs.copy())
    rewards.append(reward)
    done = terminated or truncated

env.close()
trajectory = np.array(trajectory)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Health metrics
ax = axes[0, 0]
for i, label in zip([0, 1, 2], ["Sleep Debt", "Cortisol", "Caffeine Toxicity"]):
    ax.plot(trajectory[:, i], label=label)
ax.axhline(y=0.8, color="red", linestyle=":", alpha=0.5, label="Danger zone")
ax.set_title("Health Metrics")
ax.set_ylabel("Value [0-1]")
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 1.05)

# Company metrics
ax = axes[0, 1]
ax.plot(trajectory[:, 3], label="Product Velocity", color="green")
ax.plot(trajectory[:, 4], label="Team Morale", color="purple")
ax.axhline(y=0.8, color="green", linestyle=":", alpha=0.5, label="Launch threshold")
ax.set_title("Company Metrics")
ax.set_ylabel("Value [0-1]")
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 1.05)

# Cash runway
ax = axes[1, 0]
ax.plot(trajectory[:, 5], color="orange", linewidth=2)
ax.axhline(y=0.0, color="red", linestyle="--", label="Bankruptcy")
ax.set_title("Cash Runway")
ax.set_ylabel("Value [0-1]")
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 1.05)

# Market + Crisis
ax = axes[1, 1]
ax.plot(trajectory[:, 6], label="Market Condition", color="teal")
ax.plot(trajectory[:, 7], label="Active Crisis", color="red", alpha=0.7)
ax.set_title("External Factors")
ax.set_ylabel("Value [0-1]")
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 1.05)

# Cumulative reward
ax = axes[2, 0]
ax.plot(np.cumsum(rewards), color="darkblue", linewidth=2)
ax.set_title("Cumulative Reward")
ax.set_xlabel("Day")
ax.set_ylabel("Cumulative Reward")

# Per-step reward
ax = axes[2, 1]
ax.bar(range(len(rewards)), rewards, color="steelblue", alpha=0.6)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Per-Step Reward")
ax.set_xlabel("Day")
ax.set_ylabel("Reward")

plt.suptitle("FounderEndurance-v1: Single Episode Trajectory (Trained PPO)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f"\nEpisode length: {len(rewards)} days")
print(f"Total reward: {sum(rewards):.2f}")

## 9. Compare: Random Agent vs Trained Agent

In [ ]:
def run_episodes(model, n_episodes=50, deterministic=True):
    """Run episodes and return rewards and lengths."""
    env = gym.make("FounderEndurance-v1")
    rewards, lengths = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep + 1000)
        total_r, steps = 0.0, 0
        done = False
        while not done:
            if model is None:
                action = env.action_space.sample()
            else:
                action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, _ = env.step(action)
            total_r += r
            steps += 1
            done = terminated or truncated
        rewards.append(total_r)
        lengths.append(steps)
    env.close()
    return np.array(rewards), np.array(lengths)

# Run both
random_rewards, random_lengths = run_episodes(None, n_episodes=50)
ppo_rewards, ppo_lengths = run_episodes(best_model, n_episodes=50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward comparison
ax = axes[0]
ax.boxplot([random_rewards, ppo_rewards], labels=["Random", "PPO (trained)"])
ax.set_ylabel("Total Episode Reward")
ax.set_title("Reward Distribution: Random vs PPO")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

# Survival comparison
ax = axes[1]
ax.boxplot([random_lengths, ppo_lengths], labels=["Random", "PPO (trained)"])
ax.set_ylabel("Episode Length (days)")
ax.set_title("Survival Duration: Random vs PPO")
ax.axhline(y=90, color="green", linestyle="--", alpha=0.5, label="Full 90 days")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Random  -> Mean reward: {random_rewards.mean():.1f}, Mean survival: {random_lengths.mean():.1f} days")
print(f"PPO     -> Mean reward: {ppo_rewards.mean():.1f}, Mean survival: {ppo_lengths.mean():.1f} days")

## 10. Launch TensorBoard (inline)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./tb_logs/

## 11. Download Trained Model

Download the trained model weights to your local machine.

In [ ]:
from google.colab import files

# Zip the models directory
!zip -r founder_endurance_models.zip models/
files.download("founder_endurance_models.zip")